In [2]:
import pandas as pd
import requests
from io import StringIO
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [3]:
expulsion_url = "https://www3.cde.ca.gov/demo-downloads/discipline/expulsion25.txt"

exp_r = requests.get(expulsion_url)
exp_r.raise_for_status()
exp_text = exp_r.content.decode("cp1252", errors="replace")

In [4]:
expulsion_df = pd.read_csv(
    StringIO(exp_text),
    sep="\t",
    dtype={
        "AcademicYear": "string",
        "AggregateLevel": "string",
        "CountyCode": "string",
        "DistrictCode": "string",
        "SchoolCode": "string",
        "CountyName": "string",
        "DistrictName": "string",
        "SchoolName": "string",
        "CharterYN": "string",
        "ReportingCategory": "string"
    },
    keep_default_na=False
)

expulsion_df.columns = expulsion_df.columns.str.strip()

# strip whitespace from string columns
for col in ["AcademicYear", "AggregateLevel", "CountyCode", "DistrictCode", "SchoolCode",
            "CountyName", "DistrictName", "SchoolName", "CharterYN", "ReportingCategory"]:
    if col in expulsion_df.columns:
        expulsion_df[col] = expulsion_df[col].str.strip()

In [5]:
expulsion_df

,AcademicYear,AggregateLevel,CountyCode,DistrictCode,SchoolCode,CountyName,DistrictName,SchoolName,CharterYN,ReportingCategory,...,Total Expulsions,Unduplicated Count of Students Expelled (Total),Unduplicated Count of Students Expelled (Defiance-Only),Expulsion Rate (Total),Expulsion Count Violent Incident (Injury),Expulsion Count Violent Incident (No Injury),Expulsion Count Weapons Possession,Expulsion Count Illicit Drug-Related,Expulsion Count Defiance-Only,Expulsion Count Other Reasons
0,2024-25,C,01,,,Alameda,,,All,GF,...,20,20,0,0.0,15,3,1,1,0,0
1,2024-25,C,01,,,Alameda,,,All,GM,...,47,47,0,0.0,17,9,16,5,0,0
2,2024-25,C,01,,,Alameda,,,All,GX,...,1,1,0,0.3,1,0,0,0,0,0
3,2024-25,C,01,,,Alameda,,,All,RA,...,2,2,0,0.0,2,0,0,0,0,0
4,2024-25,C,01,,,Alameda,,,All,RB,...,17,17,0,0.1,10,2,4,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225456,2024-25,T,00,,,,,,Yes,SF,...,6,6,0,0.1,2,4,0,0,0,0
225457,2024-25,T,00,,,,,,Yes,SH,...,16,16,0,0.0,7,8,1,0,0,0
225458,2024-25,T,00,,,,,,Yes,SM,...,2,2,0,0.1,1,0,0,0,0,1
225459,2024-25,T,00,,,,,,Yes,SS,...,175,173,0,0.0,52,74,25,22,0,2


In [6]:
exp_school_df = expulsion_df[expulsion_df["AggregateLevel"] == "S"].copy()

# remove rows with missing/blank school codes
exp_school_df = exp_school_df[exp_school_df["SchoolCode"] != ""].copy()

# zero-pad codes
exp_school_df["CountyCode"] = exp_school_df["CountyCode"].str.zfill(2)
exp_school_df["DistrictCode"] = exp_school_df["DistrictCode"].str.zfill(5)
exp_school_df["SchoolCode"] = exp_school_df["SchoolCode"].str.zfill(7)

# keep only schools in your list
exp_school_df = exp_school_df[exp_school_df["SchoolCode"].isin(school_codes_list)].copy()
exp_school_df = exp_school_df[exp_school_df['ReportingCategory'] == 'TA'].copy()

exp_school_df = exp_school_df.reset_index(drop=True)

In [7]:
exp_school_df = exp_school_df.drop(columns = [col for col in exp_school_df if ('Exp' in col and col != 'Total Expulsions')])
exp_school_df = exp_school_df.drop(columns = [col for col in exp_school_df.columns if col in ['AcademicYear', 'AggregateLevel', 'CountyName', 'DistrictName', 'ReportingCategory']])

In [8]:
exp_school_df

,CountyCode,DistrictCode,SchoolCode,SchoolName,CharterYN,CumulativeEnrollment,Total Expulsions
0,01,10017,0130625,Alternatives in Action,Yes,124,0
1,01,61119,0130229,Alameda High,No,1904,0
2,01,61119,0106401,Alameda Science and Technology Institute,No,213,0
3,01,61127,0130450,Albany High,No,1144,0
4,01,61143,0131177,Berkeley High,No,3303,1
...,...,...,...,...,...,...,...
1391,57,10579,0113787,Cesar Chavez Community,No,69,0
1392,58,72736,5830013,Lindhurst High,No,1390,5
1393,58,72736,5835202,Marysville High,No,1102,3
1394,58,72769,5838305,Wheatland Union High,No,1135,2


In [9]:
suspension_url = "https://www3.cde.ca.gov/demo-downloads/discipline/suspension25.txt"

sus_r = requests.get(suspension_url)
sus_r.raise_for_status()
sus_text = sus_r.content.decode("cp1252", errors="replace")

In [10]:
suspension_df = pd.read_csv(
    StringIO(sus_text),
    sep="\t",
    dtype={
        "AcademicYear": "string",
        "AggregateLevel": "string",
        "CountyCode": "string",
        "DistrictCode": "string",
        "SchoolCode": "string",
        "CountyName": "string",
        "DistrictName": "string",
        "SchoolName": "string",
        "CharterYN": "string",
        "ReportingCategory": "string"
    },
    keep_default_na=False
)

suspension_df.columns = suspension_df.columns.str.strip()

# strip whitespace from string columns
for col in ["AcademicYear", "AggregateLevel", "CountyCode", "DistrictCode", "SchoolCode",
            "CountyName", "DistrictName", "SchoolName", "CharterYN", "ReportingCategory"]:
    if col in suspension_df.columns:
        suspension_df[col] = suspension_df[col].str.strip()

In [11]:
sus_school_df = suspension_df[suspension_df["AggregateLevel"] == "S"].copy()

# remove rows with missing/blank school codes
sus_school_df = sus_school_df[sus_school_df["SchoolCode"] != ""].copy()

# zero-pad codes
sus_school_df["CountyCode"] = sus_school_df["CountyCode"].str.zfill(2)
sus_school_df["DistrictCode"] = sus_school_df["DistrictCode"].str.zfill(5)
sus_school_df["SchoolCode"] = sus_school_df["SchoolCode"].str.zfill(7)

# keep only schools in your list
sus_school_df = sus_school_df[sus_school_df["SchoolCode"].isin(school_codes_list)].copy()
sus_school_df = sus_school_df[sus_school_df['ReportingCategory'] == 'TA'].copy()

sus_school_df = sus_school_df.reset_index(drop=True)

In [12]:
sus_school_df = sus_school_df.drop(columns=[col for col in sus_school_df.columns if col not in ['SchoolCode', 'Total Suspensions']])

In [13]:
school_df = exp_school_df.merge(sus_school_df, on='SchoolCode')
school_df = school_df[['SchoolCode', 'CumulativeEnrollment', 'Total Expulsions', 'Total Suspensions']]

In [15]:
school_df

,SchoolCode,CumulativeEnrollment,Total Expulsions,Total Suspensions
0,0130625,124,0,5
1,0130229,1904,0,24
2,0106401,213,0,0
3,0130450,1144,0,9
4,0131177,3303,1,101
...,...,...,...,...
1391,0113787,69,0,2
1392,5830013,1390,5,257
1393,5835202,1102,3,103
1394,5838305,1135,2,59


In [ ]:
school_df.to_csv('../cleaned_data/suspension_expulsion_data.csv')